In [ ]:
# Cell 1: Imports and Setup
import time
import logging
from datetime import datetime
from synthetic_data_generator import AmazonUserDataGenerator
from s3_uploader import S3Uploader
import config

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

# Cell 2: Initialize Components
generator = AmazonUserDataGenerator(batch_size=config.BATCH_SIZE)
uploader = S3Uploader()

# Test S3 connection
if not uploader.test_connection():
    raise Exception("S3 connection failed. Check credentials.")

# Cell 3: Single Batch Test
def generate_and_upload_single_batch():
    """Generate one batch and upload to S3"""
    
    logger.info("🔄 Generating synthetic data batch...")
    df = generator.generate_with_timestamp()
    
    logger.info(f"✅ Generated {len(df)} users")
    print(df.head())
    
    # Upload to S3
    logger.info("📤 Uploading to S3...")
    
    # Option 1: Upload as separate batch file
    s3_key = uploader.upload_dataframe(df)
    
    # Option 2: Append to master file
    uploader.append_to_master_file(df)
    
    return df

# Test single batch
test_df = generate_and_upload_single_batch()

# Cell 4: Continuous Generation (Every 1 Minute)
def run_continuous_generation(duration_minutes=60):
    """
    Run continuous data generation
    
    Args:
        duration_minutes: How long to run (default 60 minutes)
    """
    
    logger.info(f"🚀 Starting continuous generation for {duration_minutes} minutes")
    logger.info(f"📊 Batch size: {config.BATCH_SIZE} users")
    logger.info(f"⏱️  Interval: {config.GENERATION_INTERVAL} seconds")
    
    start_time = time.time()
    batch_count = 0
    
    try:
        while True:
            # Check if duration exceeded
            elapsed = (time.time() - start_time) / 60
            if elapsed >= duration_minutes:
                logger.info(f"⏰ Duration limit reached ({duration_minutes} min)")
                break
            
            batch_count += 1
            logger.info(f"\n{'='*50}")
            logger.info(f"📦 BATCH #{batch_count} - Elapsed: {elapsed:.1f} min")
            logger.info(f"{'='*50}")
            
            # Generate and upload
            df = generator.generate_with_timestamp()
            logger.info(f"✅ Generated {len(df)} users")
            
            # Upload to S3 (append to master file)
            uploader.append_to_master_file(df)
            
            # Wait for next interval
            logger.info(f"⏸️  Waiting {config.GENERATION_INTERVAL} seconds...\n")
            time.sleep(config.GENERATION_INTERVAL)
            
    except KeyboardInterrupt:
        logger.info("\n⛔ Generation stopped by user")
    
    logger.info(f"\n🎉 Completed! Total batches: {batch_count}")
    logger.info(f"📊 Total users generated: {batch_count * config.BATCH_SIZE}")

# Cell 5: Run the Generator
# Run for 60 minutes (generates 60 batches of 10 users = 600 users)
run_continuous_generation(duration_minutes=60)

# Or run indefinitely (stop manually with Kernel > Interrupt)
# run_continuous_generation(duration_minutes=float('inf'))